[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/14.%20The%20Vessel%20Stowage%20Planning%20Problem%20%28Export%29/P14-Tier-9_executed.ipynb)

---


# The Vessel Stowage Planning Problem (Export) — Tier 9
## QUBO Formulation and a QAOA-Inspired Demonstration

### Purpose and scope

This notebook introduces a small export-stowage example, expresses its binary placement decisions as a Quadratic Unconstrained Binary Optimization (QUBO) model, and follows the cost-layer/mixer-layer sequence associated with the Quantum Approximate Optimization Algorithm (QAOA). The example contains five containers, six candidate slots, and one binary decision variable for each container-slot pair. The notebook builds the model, converts it to an Ising representation, samples binary strings, decodes the best sample observed, and checks the encoded assignment rules.

The circuit code is an educational **product-state approximation**, not a full quantum-computer simulator and not a faithful implementation of QAOA. It stores each qubit separately, so it cannot represent entanglement or the pairwise phase interactions in the QUBO cost Hamiltonian. Its samples illustrate the circuit structure and binary objective evaluation; they do not establish quantum performance, quantum advantage, or an optimal stowage plan.

The stowage objective is a deliberately simplified proxy. It combines destination discharge-order costs with a bay-level weight-distribution penalty. It does not represent tier geometry, blocking relationships, vessel hydrostatics, hazardous-cargo segregation, reefer connections, or operational loading constraints. The weight proxy scores each candidate placement using the existing weight recorded for its bay plus that container's weight; it does not jointly calculate the final distribution of all selected placements. These assumptions are essential when interpreting the output.

### Inputs and outputs

The inputs are container attributes, candidate slot identifiers, discharge-order values, baseline bay weights, and penalty and cost weights. The output is the lowest-scoring sampled bit string encountered during the stated number of iterations, together with its decoded placements, constraint checks, and sampling history. A sampled string can violate assignment rules, so the checks are part of interpreting the result rather than optional presentation.

### Learning objectives

By the end, you should be able to identify the binary decision variables, understand how assignment rules become QUBO penalties, connect a binary QUBO to an Ising energy expression, describe the cost-layer/mixer-layer sequence, and explain what the product-state circuit can and cannot demonstrate.

## 1. Binary decisions and the QUBO model

A binary decision variable records whether a particular container is assigned to a particular slot. For container \(i\) and slot \(j\), define

$$
x_{i,j} =
\begin{cases}
1, & \text{if container } i \text{ is assigned to slot } j,\\
0, & \text{otherwise.}
\end{cases}
$$

There are \(n_c n_s\) such variables for \(n_c\) containers and \(n_s\) slots. The code flattens them in container-major order: the variable at index \(i n_s+j\) is \(x_{i,j}\). A QUBO objective is a quadratic function of binary variables. This notebook uses the matrix convention

$$
E(x) = x^{\mathsf{T}} Q x + \ell^{\mathsf{T}}x,
\qquad x \in \{0,1\}^{n_c n_s},
$$

where \(Q\) is symmetric and \(\ell\) contains linear coefficients. With this convention, a symmetric off-diagonal pair \(Q_{a,b}=Q_{b,a}=c\) contributes \(2c x_a x_b\), because both entries appear in \(x^{\mathsf{T}}Qx\). Diagonal entries contribute \(Q_{a,a}x_a^2=Q_{a,a}x_a\), since a binary variable satisfies \(x_a^2=x_a\). The method encodes constraints by adding penalties to the objective rather than passing them separately to an optimizer.

### Assignment and slot-capacity penalties

Each container must occupy exactly one slot. For container \(i\), let \(s_i=\sum_j x_{i,j}\). The penalty is

$$
A(1-s_i)^2
= A - A\sum_j x_{i,j} + 2A\sum_{j<k}x_{i,j}x_{i,k}.
$$

The constant \(A\) does not affect which assignment minimizes the objective, so it is omitted from the QUBO. The remaining terms give each assignment variable a linear coefficient \(-A\), and give each pair of slots for the same container a quadratic penalty. With the symmetric-matrix convention, the two matrix entries for such a pair are each \(A\), producing the total coefficient \(2A\).

Each slot can hold at most one container. For each pair of different containers assigned to the same slot, the model adds a penalty \(A x_{i,j}x_{k,j}\). The two symmetric matrix entries are each \(A/2\), so their combined contribution is \(A x_{i,j}x_{k,j}\). A slot is allowed to remain unused.

### Proxy costs and the complete encoded objective

For a container \(i\) and a candidate slot \(j\) in bay \(b(j)\), the code assigns the placement cost

$$
c_{i,j}
= o_{d(i)} P_o
+ \lambda_w\left|W_{b(j)}+w_i-\frac{W_{\mathrm{total}}}{22}\right|.
$$

Here \(o_{d(i)}\) is the discharge-order value for the container's destination, \(P_o\) is the overstowage proxy weight, \(W_{b(j)}\) is the recorded baseline weight for the slot's bay, \(w_i\) is the container weight, \(W_{\mathrm{total}}\) is the sum of the example's container weights, and \(\lambda_w\) is the stability proxy weight. The divisor 22 is part of this example's stated proxy; the resulting quantity is not a computed target for the final combined bay weights.

Writing the encoded objective in terms of the placement variables makes the role of each term explicit:

$$
E(x)
= \sum_{i,j} c_{i,j}x_{i,j}
 - A\sum_i\sum_j x_{i,j}
 + 2A\sum_i\sum_{j<k}x_{i,j}x_{i,k}
 + A\sum_j\sum_{i<k}x_{i,j}x_{k,j}.
$$

The first term scores placements. The next two terms encode exactly-one assignment, with the assignment-independent constant omitted. The final term penalizes two containers sharing a slot. The implementation constructs these terms in `formulate_stowage_qubo`; the matrix and linear-vector convention is implemented by `QUBOProblem.evaluate`. These costs are modeling proxies, not a complete physical model of stowage.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")


@dataclass
class QuantumBit:
    """A single-qubit state used by the product-state teaching simulator."""

    id: int
    alpha: complex = 1.0 + 0.0j
    beta: complex = 0.0 + 0.0j

    def normalize(self) -> None:
        """Keep the state vector at unit length after a gate operation."""
        norm = math.sqrt(abs(self.alpha) ** 2 + abs(self.beta) ** 2)
        if norm == 0:
            raise ValueError("A qubit state cannot have zero norm.")
        self.alpha /= norm
        self.beta /= norm

    def measure(self) -> int:
        """Sample the computational-basis outcome using its Born probability."""
        probability_zero = abs(self.alpha) ** 2
        return 0 if random.random() < probability_zero else 1

    def __str__(self) -> str:
        return f"|ψ⟩ = {self.alpha:.3f}|0⟩ + {self.beta:.3f}|1⟩"


@dataclass
class QuantumGate:
    """A single-qubit gate represented by a 2-by-2 matrix."""

    name: str
    matrix: np.ndarray

    def apply(
        self,
        qubits: List[QuantumBit],
        target_indices: Optional[List[int]] = None,
    ) -> None:
        """Apply the matrix independently to each selected qubit."""
        if self.matrix.shape != (2, 2):
            raise ValueError("This product-state simulator supports only 2-by-2 gates.")
        targets = target_indices if target_indices is not None else list(range(len(qubits)))
        for index in targets:
            if not 0 <= index < len(qubits):
                raise IndexError(f"Qubit index {index} is out of range.")
            qubit = qubits[index]
            state_vector = np.array([qubit.alpha, qubit.beta], dtype=complex)
            new_state = self.matrix @ state_vector
            qubit.alpha, qubit.beta = new_state
            qubit.normalize()


@dataclass
class QuantumCircuit:
    """A sequence of single-qubit gates for independent qubit states."""

    qubits: List[QuantumBit] = field(default_factory=list)
    gates: List[Tuple[QuantumGate, List[int]]] = field(default_factory=list)

    def add_qubit(self, qubit: QuantumBit) -> None:
        self.qubits.append(qubit)

    def add_gate(self, gate: QuantumGate, target_indices: List[int]) -> None:
        self.gates.append((gate, target_indices))

    def execute(self) -> None:
        for gate, target_indices in self.gates:
            gate.apply(self.qubits, target_indices)

    def measure_all(self) -> List[int]:
        return [qubit.measure() for qubit in self.qubits]


@dataclass
class QUBOProblem:
    """A binary quadratic objective with symmetric-matrix convention."""

    n_variables: int
    Q_matrix: np.ndarray
    linear_terms: Optional[np.ndarray] = None

    def __post_init__(self) -> None:
        if self.Q_matrix.shape != (self.n_variables, self.n_variables):
            raise ValueError("Q_matrix dimensions must match n_variables.")
        if not np.allclose(self.Q_matrix, self.Q_matrix.T):
            raise ValueError("Q_matrix must be symmetric.")
        if self.linear_terms is None:
            self.linear_terms = np.zeros(self.n_variables)
        elif self.linear_terms.shape != (self.n_variables,):
            raise ValueError("linear_terms must contain one value per variable.")

    def evaluate(self, x: np.ndarray) -> float:
        """Evaluate the QUBO energy of a binary vector."""
        if x.shape != (self.n_variables,):
            raise ValueError("The solution vector has the wrong number of variables.")
        if not np.all(np.isin(x, [0, 1])):
            raise ValueError("QUBO solutions must be binary.")
        return float(x @ self.Q_matrix @ x + self.linear_terms @ x)

    def to_ising(self) -> Tuple[np.ndarray, np.ndarray, float]:
        """Return J, h, and c for E(z) = z.T @ J @ z + h.T @ z + c."""
        # The substitution x = (1 + z) / 2 converts binary variables to spins.
        J = self.Q_matrix / 4
        h = self.Q_matrix.sum(axis=0) / 2 + self.linear_terms / 2
        constant = float(self.Q_matrix.sum() / 4 + self.linear_terms.sum() / 2)
        return J, h, constant

## 2. QAOA structure and simulator limits

A bit string is one possible assignment of 0s and 1s to all decision variables. QAOA uses quantum states to assign amplitudes to such bit strings and alternates two operations. The **cost unitary** gives each computational-basis state a phase determined by its objective value. The **mixer unitary** changes amplitudes so that later measurements can sample different bit strings. For a cost Hamiltonian \(H_C\), the standard mixer Hamiltonian \(H_M=\sum_{k=1}^{n}X_k\), and depth \(p\), the idealized sequence is

$$
|\psi(\boldsymbol{\gamma},\boldsymbol{\beta})\rangle
= U_M(\beta_p)U_C(\gamma_p)\cdots U_M(\beta_1)U_C(\gamma_1)|+\rangle^{\otimes n},
$$

with

$$
U_C(\gamma)=e^{-i\gamma H_C},
\qquad
U_M(\beta)=e^{-i\beta H_M}.
$$

Here \(n\) is the number of binary variables, \(|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}\), and \(\boldsymbol{\gamma}\) and \(\boldsymbol{\beta}\) are the cost and mixer angle parameters. The cost Hamiltonian is diagonal in the computational basis: its energy for a basis state is the modeled objective for the corresponding bit string. The mixer does not enforce the assignment constraints in this example; those rules are represented by penalties in the objective.

For the binary-to-spin substitution \(x=(1+z)/2\), where each spin \(z_k\in\{-1,1\}\), `QUBOProblem.to_ising` returns the coefficients in

$$
E(z)=z^{\mathsf{T}}Jz+h^{\mathsf{T}}z+c.
$$

The diagonal terms of \(J\) multiply \(z_k^2=1\), so they contribute only a constant phase in a cost unitary. Off-diagonal entries of \(J\) encode pairwise spin interactions and require multi-qubit operations in a circuit representation.

This notebook retains the cost-layer/mixer-layer teaching sequence but uses a product-state simulator. Its cost layer applies only single-qubit rotations derived from the linear Ising terms \(h\); it omits pairwise interactions. Each qubit is measured independently, so the simulator cannot represent a joint state, entanglement, or correlations generated by multi-qubit gates. The parameter perturbation is a simple demonstration mechanism, not a standard QAOA parameter optimizer. Consequently, the results are **QAOA-inspired samples**, not QAOA solutions or evidence of quantum optimization performance.

The \(n\)-bit search space contains \(2^n\) bit strings. This is a count of possible assignments; it does not by itself imply that any particular algorithm examines all assignments or achieves a speedup.

In [ ]:
class QuantumOptimizer:
    """Build a QUBO and demonstrate a product-state QAOA-inspired sequence."""

    def __init__(self, n_qubits: int, optimization_level: int = 1):
        self.n_qubits = n_qubits
        self.optimization_level = optimization_level
        self._initialize_quantum_gates()

    def _initialize_quantum_gates(self) -> None:
        """Create the single-qubit gates used by the demonstration circuit."""
        self.X_gate = QuantumGate(
            "X", np.array([[0, 1], [1, 0]], dtype=complex)
        )
        self.H_gate = QuantumGate(
            "H", np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
        )

        def rx(theta: float) -> QuantumGate:
            matrix = np.array(
                [
                    [math.cos(theta / 2), -1j * math.sin(theta / 2)],
                    [-1j * math.sin(theta / 2), math.cos(theta / 2)],
                ],
                dtype=complex,
            )
            return QuantumGate(f"Rx({theta:.3f})", matrix)

        def rz(theta: float) -> QuantumGate:
            matrix = np.array(
                [
                    [np.exp(-1j * theta / 2), 0],
                    [0, np.exp(1j * theta / 2)],
                ],
                dtype=complex,
            )
            return QuantumGate(f"Rz({theta:.3f})", matrix)

        self.Rx_gate = rx
        self.Rz_gate = rz

    def create_superposition_state(self, n_qubits: int) -> QuantumCircuit:
        """Prepare independent qubits in the |+> state."""
        circuit = QuantumCircuit()
        for index in range(n_qubits):
            circuit.add_qubit(QuantumBit(index))
        for index in range(n_qubits):
            circuit.add_gate(self.H_gate, [index])
        return circuit

    def formulate_stowage_qubo(
        self,
        containers: List[Dict[str, Any]],
        slots: List[str],
        constraints: Dict[str, Any],
    ) -> QUBOProblem:
        """Encode assignment rules and proxy costs in a symmetric QUBO."""
        n_containers = len(containers)
        n_slots = len(slots)
        n_variables = n_containers * n_slots
        Q = np.zeros((n_variables, n_variables), dtype=float)
        linear = np.zeros(n_variables, dtype=float)
        penalty = float(constraints["constraint_penalty"])

        def variable_index(container_index: int, slot_index: int) -> int:
            return container_index * n_slots + slot_index

        # Each container must use exactly one slot: A(1 - sum_j x_ij)^2,
        # with the assignment-independent constant A omitted.
        for i, container in enumerate(containers):
            discharge_order = constraints["discharge_order"][container["destination"]]
            for j, slot in enumerate(slots):
                index = variable_index(i, j)
                linear[index] -= penalty

                bay = int(slot[:2])
                ideal_weight = constraints["total_weight"] / 22
                current_weight = constraints["weight_by_bay"].get(bay, 0)
                projected_weight = current_weight + container["weight"]
                stability_cost = abs(projected_weight - ideal_weight)
                Q[index, index] += (
                    discharge_order * constraints["overstowage_penalty"]
                    + stability_cost * constraints["stability_weight"]
                )

            for j1 in range(n_slots):
                for j2 in range(j1 + 1, n_slots):
                    index1 = variable_index(i, j1)
                    index2 = variable_index(i, j2)
                    # A in both symmetric positions contributes 2A*x1*x2.
                    Q[index1, index2] += penalty
                    Q[index2, index1] += penalty

        # A slot may be unused, but two containers may not share it.
        # The two symmetric entries sum to one penalty for each conflicting pair.
        for j in range(n_slots):
            for i1 in range(n_containers):
                for i2 in range(i1 + 1, n_containers):
                    index1 = variable_index(i1, j)
                    index2 = variable_index(i2, j)
                    Q[index1, index2] += penalty / 2
                    Q[index2, index1] += penalty / 2

        return QUBOProblem(n_variables, Q, linear)

    def qaoa_optimization(
        self,
        qubo: QUBOProblem,
        p_layers: int = 2,
        max_iterations: int = 100,
    ) -> Dict[str, Any]:
        """Sample QUBO bit strings using the stated product-state approximation."""
        if p_layers < 1 or max_iterations < 1:
            raise ValueError("p_layers and max_iterations must be positive.")

        print(
            "Running the product-state QAOA-inspired demonstration "
            f"with {p_layers} layers..."
        )
        _, h, _ = qubo.to_ising()
        gamma_params = np.random.uniform(0, 2 * np.pi, p_layers)
        beta_params = np.random.uniform(0, np.pi, p_layers)
        best_solution = None
        best_value = float("inf")
        optimization_history = []

        for iteration in range(max_iterations):
            circuit = self.create_superposition_state(qubo.n_variables)
            for layer in range(p_layers):
                gamma = gamma_params[layer]
                beta = beta_params[layer]

                # Pairwise Ising interactions are omitted in this simulator.
                # This rotation implements exp(-i * gamma * h_k * Z_k).
                for index, field_value in enumerate(h):
                    if abs(field_value) > 1e-10:
                        circuit.add_gate(
                            self.Rz_gate(2 * gamma * field_value), [index]
                        )
                for index in range(qubo.n_variables):
                    circuit.add_gate(self.Rx_gate(2 * beta), [index])

            circuit.execute()
            solution = np.array(circuit.measure_all(), dtype=int)
            value = qubo.evaluate(solution)
            if value < best_value:
                best_value = value
                best_solution = solution.copy()

            optimization_history.append(
                {
                    "iteration": iteration,
                    "best_value": best_value,
                    "current_value": value,
                    "gamma_params": gamma_params.copy(),
                    "beta_params": beta_params.copy(),
                }
            )

            # This perturbation is a transparent teaching device, not a
            # claim to implement a standard variational optimizer.
            if iteration % 10 == 0 and iteration > 0:
                gamma_params = np.clip(
                    gamma_params + np.random.normal(0, 0.1, p_layers),
                    0,
                    2 * np.pi,
                )
                beta_params = np.clip(
                    beta_params + np.random.normal(0, 0.1, p_layers),
                    0,
                    np.pi,
                )

        return {
            "best_solution": best_solution,
            "best_value": best_value,
            "optimization_history": optimization_history,
            "final_gamma_params": gamma_params,
            "final_beta_params": beta_params,
            "qubo_problem": qubo,
        }

    def decode_solution(
        self,
        solution: np.ndarray,
        containers: List[Dict[str, Any]],
        slots: List[str],
    ) -> List[Dict[str, Any]]:
        """Convert a binary vector into one record per container."""
        expected_length = len(containers) * len(slots)
        if len(solution) != expected_length:
            raise ValueError("The solution length does not match the instance.")

        placement_plan = []
        for i, container in enumerate(containers):
            selected_slots = [
                j for j in range(len(slots))
                if solution[i * len(slots) + j] == 1
            ]
            if len(selected_slots) == 1:
                placement_plan.append(
                    {
                        "container_id": container["id"],
                        "container": container,
                        "slot": slots[selected_slots[0]],
                        "placed": True,
                    }
                )
            else:
                placement_plan.append(
                    {
                        "container_id": container["id"],
                        "container": container,
                        "slot": None,
                        "placed": False,
                    }
                )
        return placement_plan

## 3. Define the example instance

The slot identifiers begin with their bay numbers, so `slot[:2]` identifies the bay for this example. The existing bay weights are baseline data; the placement cost uses that baseline plus the candidate container's weight. This is an additive proxy: it scores each container-slot choice separately and does not jointly model the final weight distribution created by all selected placements. The container types are retained as instance data, but this QUBO does not encode type-specific rules such as hazardous-cargo segregation or reefer connections.

In [ ]:
print("=== Vessel Stowage QUBO Demonstration ===")

containers_quantum = [
    {"id": "Q_CONT_1", "weight": 20, "destination": "Rotterdam", "type": "standard"},
    {"id": "Q_CONT_2", "weight": 25, "destination": "Singapore", "type": "hazardous"},
    {"id": "Q_CONT_3", "weight": 18, "destination": "Dubai", "type": "reefer"},
    {"id": "Q_CONT_4", "weight": 22, "destination": "Antwerp", "type": "standard"},
    {"id": "Q_CONT_5", "weight": 30, "destination": "Rotterdam", "type": "oversized"},
]

slots_quantum = [
    "140284", "140286",
    "160284", "160286",
    "180284", "180286",
]

constraints_quantum = {
    "discharge_order": {
        "Rotterdam": 0,
        "Antwerp": 1,
        "Singapore": 2,
        "Dubai": 3,
    },
    "overstowage_penalty": 100,
    "constraint_penalty": 1000,
    "stability_weight": 10,
    "total_weight": sum(container["weight"] for container in containers_quantum),
    "weight_by_bay": {14: 50, 16: 60, 18: 40},
}

n_variables = len(containers_quantum) * len(slots_quantum)
print(f"Containers: {len(containers_quantum)}")
print(f"Candidate slots: {len(slots_quantum)}")
print(f"Binary variables: {n_variables}")
print(f"Possible binary strings: 2^{n_variables}")

quantum_optimizer = QuantumOptimizer(
    n_qubits=n_variables,
    optimization_level=2,
)
print(
    "Product-state simulator variables: "
    f"{quantum_optimizer.n_qubits}"
)

## 4. Form and inspect the QUBO

The matrix is indexed in container-major order: variable \(i n_s+j\) represents assigning container \(i\) to slot \(j\). The diagonal contains per-assignment proxy costs, while off-diagonal entries penalize incompatible pairs. Linear coefficients and same-container quadratic terms encode the exactly-one assignment condition; same-slot quadratic terms encode slot capacity. Because the model uses a symmetric matrix, an off-diagonal pair contributes the sum of its two entries when evaluated as \(x^{\mathsf{T}}Qx\).

The binary-to-spin conversion uses \(x_k=(1+z_k)/2\), with \(z_k\in\{-1,1\}\). This is a change of variables, not a new optimization method: every binary string corresponds to exactly one spin string, and the energy is preserved when the constant term is included.

In [ ]:
qubo_problem = quantum_optimizer.formulate_stowage_qubo(
    containers_quantum,
    slots_quantum,
    constraints_quantum,
)

print("=== QUBO Formulation ===")
print(f"Variables: {qubo_problem.n_variables}")
print(f"Q matrix dimensions: {qubo_problem.Q_matrix.shape}")
print(f"Nonzero Q entries: {np.count_nonzero(qubo_problem.Q_matrix)}")
print(f"Q matrix symmetric: {np.allclose(qubo_problem.Q_matrix, qubo_problem.Q_matrix.T)}")

display_size = min(10, qubo_problem.n_variables)
print(f"\nTop-left {display_size} by {display_size} block of Q:")
for row_index in range(display_size):
    values = []
    for column_index in range(display_size):
        value = qubo_problem.Q_matrix[row_index, column_index]
        values.append(f"{value:7.1f}" if abs(value) > 0.1 else "      .")
    print(f"Row {row_index:2d}: " + " ".join(values))

J, h, constant = qubo_problem.to_ising()
print("\nIsing conversion dimensions:")
print(f"J: {J.shape}; h: {h.shape}; constant: {constant:.2f}")

# Check the algebraic binary-to-spin conversion on a sample bit string.
sample_bits = np.zeros(qubo_problem.n_variables, dtype=int)
sample_energy_qubo = qubo_problem.evaluate(sample_bits)
sample_spins = 2 * sample_bits - 1
sample_energy_ising = float(sample_spins @ J @ sample_spins + h @ sample_spins + constant)
assert math.isclose(sample_energy_qubo, sample_energy_ising, rel_tol=1e-10, abs_tol=1e-10)
print("Binary and Ising energy agree for the all-zero test vector.")

## 5. Run the QAOA-inspired sampling demonstration

At each iteration, the simulator prepares independent qubits in \(|+\rangle\), applies the stated single-qubit cost rotations and mixer rotations for each layer, measures a bit string, and scores that string with the full QUBO objective. In the code, `gamma_params` and `beta_params` hold the layer angles, `circuit.execute()` applies the queued gates, and `QUBOProblem.evaluate` computes the sampled string's energy. The current parameter perturbation is not an objective-directed variational optimization step.

The circuit omits pairwise Ising interactions, so evaluating samples with the full objective does not make the sampling process a full QAOA implementation. A low objective value is not, by itself, proof that the assignment is feasible: inspect the decoded plan and its constraint checks. If two samples have equal objective values, the stored result is one of the best samples observed, not necessarily a unique best assignment.

In [ ]:
start_time = time.time()
qaoa_results = quantum_optimizer.qaoa_optimization(
    qubo_problem,
    p_layers=3,
    max_iterations=50,
)
simulation_time = time.time() - start_time

placement_plan = quantum_optimizer.decode_solution(
    qaoa_results["best_solution"],
    containers_quantum,
    slots_quantum,
)

print("\n=== Best Sample Observed ===")
print(f"Best QUBO objective: {qaoa_results['best_value']:.2f}")
print(f"Elapsed simulation time: {simulation_time:.3f} seconds")
for placement in placement_plan:
    if placement["placed"]:
        container = placement["container"]
        print(
            f"{placement['container_id']} -> {placement['slot']} "
            f"(weight: {container['weight']} tons; "
            f"destination: {container['destination']})"
        )
    else:
        print(f"{placement['container_id']} -> no unique slot selected")

selected_counts = [
    int(
        qaoa_results["best_solution"][i * len(slots_quantum):
                                        (i + 1) * len(slots_quantum)].sum()
    )
    for i in range(len(containers_quantum))
]
slot_counts = [
    int(
        sum(
            qaoa_results["best_solution"][i * len(slots_quantum) + j]
            for i in range(len(containers_quantum))
        )
    )
    for j in range(len(slots_quantum))
]
all_containers_assigned_once = all(count == 1 for count in selected_counts)
slot_capacity_satisfied = all(count <= 1 for count in slot_counts)
print("\nConstraint checks:")
print(f"Every container assigned exactly once: {all_containers_assigned_once}")
print(f"No slot assigned to multiple containers: {slot_capacity_satisfied}")
print(
    "Feasible under the encoded assignment rules: "
    f"{all_containers_assigned_once and slot_capacity_satisfied}"
)

## 6. Inspect the sampling history

The stored best objective is the lowest full-QUBO score encountered up to each iteration. It can stay unchanged for many iterations because measurements are stochastic and the parameter update is deliberately simple. The plotted values describe this simulator run only; they are not a convergence guarantee for QAOA or evidence of a global optimum. The parameter plot shows the final stored angle arrays; it does not demonstrate that an objective-based optimizer found optimal angles.

In [ ]:
optimization_history = qaoa_results["optimization_history"]
iterations = [record["iteration"] for record in optimization_history]
best_values = [record["best_value"] for record in optimization_history]
current_values = [record["current_value"] for record in optimization_history]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(iterations, best_values, label="Best objective so far")
axes[0].plot(iterations, current_values, linestyle="--", alpha=0.7, label="Current sample")
axes[0].set_xlabel("Sampling iteration")
axes[0].set_ylabel("QUBO objective (model units)")
axes[0].set_title("Sampled objective history")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

layer_numbers = np.arange(1, len(qaoa_results["final_gamma_params"]) + 1)
axes[1].plot(
    layer_numbers,
    qaoa_results["final_gamma_params"],
    marker="o",
    label="Gamma (γ)",
)
axes[1].plot(
    layer_numbers,
    qaoa_results["final_beta_params"],
    marker="s",
    label="Beta (β)",
)
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Angle (radians)")
axes[1].set_title("Final demonstration parameters")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7. Single-qubit superposition and measurement

A qubit state is represented as \(\alpha|0\rangle+\beta|1\rangle\), with \(|\alpha|^2+|\beta|^2=1\). Measurement in the computational basis returns 0 with probability \(|\alpha|^2\) and 1 with probability \(|\beta|^2\). Applying a Hadamard gate to \(|0\rangle\) creates equal measurement probabilities because the resulting amplitudes have equal magnitude.

The simulator stores qubits independently. Its measurements therefore cannot demonstrate entanglement or correlations generated by multi-qubit gates. The frequency plot below is a finite-sample illustration; observed frequencies need not be exactly equal. Each bar represents the fraction of the stated number of independent circuit measurements that produced that three-bit outcome.

In [ ]:
print("=== Independent-Qubit Superposition Demonstration ===")
superposition_circuit = quantum_optimizer.create_superposition_state(3)
superposition_circuit.execute()

for index, qubit in enumerate(superposition_circuit.qubits):
    print(f"Qubit {index}: {qubit}")
    print(f"  Probability of 0: {abs(qubit.alpha) ** 2:.3f}")
    print(f"  Probability of 1: {abs(qubit.beta) ** 2:.3f}")

measurement_counts = defaultdict(int)
n_measurements = 100
for _ in range(n_measurements):
    circuit = quantum_optimizer.create_superposition_state(3)
    circuit.execute()
    measurement_counts[tuple(circuit.measure_all())] += 1

states = [format(value, "03b") for value in range(8)]
frequencies = [
    measurement_counts[tuple(int(bit) for bit in state)] / n_measurements
    for state in states
]
plt.figure(figsize=(9, 4))
plt.bar(states, frequencies, color="steelblue")
plt.xlabel("Measured three-bit state")
plt.ylabel("Observed frequency")
plt.title("Finite-sample measurements of three independent |+> qubits")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.show()

print(
    "The simulator applies single-qubit gates only; it does not represent "
    "entanglement or multi-qubit interference."
)

## 8. Interpreting the model and its computational scale

With \(n_c\) containers and \(n_s\) slots, this encoding uses \(n=n_c n_s\) binary variables and has \(2^n\) possible bit strings before feasibility rules are considered. This is a count of candidate strings, not a runtime estimate. The product-state circuit stores each qubit separately, so its memory use does not represent the resource requirements of an exact joint-state simulation.

The QUBO penalties encourage valid assignments but do not guarantee that every sampled string is feasible. Penalty magnitudes must be chosen with respect to the scale of the objective terms; feasibility checks remain necessary. In this example, the weight term is additive by placement and does not calculate the combined final bay weights. The destination-order term is a proxy and does not calculate actual overstowage without a model of vertical stacking and discharge access.

For this assignment encoding, a feasible solution requires exactly one selected slot for each container and at most one container per slot. The checks in the sampling cell test these two conditions directly on the best observed bit string. They do not validate physical stowage feasibility beyond those encoded assignment rules.

A real deployment would require a validated vessel geometry, operational and safety constraints, calibrated objective units, and an implementation that represents the pairwise cost interactions. None of those capabilities is supplied by the product-state demonstration in this notebook.

In [ ]:
n_containers = len(containers_quantum)
n_slots = len(slots_quantum)
n_binary_variables = qubo_problem.n_variables
number_of_bit_strings = 2 ** n_binary_variables

print("=== Instance Summary ===")
print(f"Containers: {n_containers}")
print(f"Candidate slots: {n_slots}")
print(f"Binary variables: {n_binary_variables}")
print(f"Possible binary strings: 2^{n_binary_variables} = {number_of_bit_strings:,}")
print(f"Samples drawn in the demonstration: {len(optimization_history)}")
print(f"Best sampled QUBO objective: {qaoa_results['best_value']:.2f}")
print(
    "Assignment rules satisfied by the best sample: "
    f"{all_containers_assigned_once and slot_capacity_satisfied}"
)
print(
    "Interpretation: these are properties of this model and simulator run, "
    "not evidence of a quantum speedup or a globally optimal plan."
)

## Conclusion

This notebook follows a QUBO-based modeling sequence: define binary container-slot decisions, add quadratic penalties for assignment rules, evaluate binary solutions, convert the binary objective to an Ising energy expression, and introduce the cost/mixer structure associated with QAOA. The QUBO-to-Ising conversion is included with an explicit convention so that its energy can be checked against the binary objective.

The implementation is intentionally limited. Its independent-qubit simulator omits pairwise cost interactions and entanglement, and its parameter perturbation is not a standard QAOA optimizer. Its output is therefore a programming and modeling demonstration, not a quantum optimization result. The stowage objective is a proxy rather than an operationally complete vessel model.

The central modeling lesson is that encoding a problem as a QUBO is only one part of an optimization workflow. The decision variables, cost terms, penalties, physical assumptions, sampling implementation, and feasibility checks must all agree before a computed assignment can be interpreted as a practical stowage plan.